In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
import time
import warnings
from tabulate import tabulate
warnings.filterwarnings('ignore')

# ============================================================================
# 載入資料
# ============================================================================
print("=" * 80)
print("載入資料集...")
print("=" * 80)

# 定義欄位名稱
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num',
    'marital-status', 'occupation', 'relationship', 'race', 'sex',
    'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income'
]

train_url = "adult.data"
test_url = "adult.test"

# 載入訓練集
train_df = pd.read_csv(train_url, names=column_names, sep=',\s*', 
                       engine='python', na_values='?')

# 載入測試集
test_df = pd.read_csv(test_url, names=column_names, sep=',\s*', 
                      engine='python', na_values='?', skiprows=1)

# 清理資料（去除空白與句點）
train_df = train_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
test_df = test_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
test_df['income'] = test_df['income'].str.replace('.', '', regex=False).str.strip()

print(f"\n原始訓練集大小: {train_df.shape}")
print(f"原始測試集大小: {test_df.shape}")
# ============================================================================
# 資料前置處理
# ============================================================================
print("\n" + "=" * 80)
print("資料前置處理")
print("=" * 80)

# 複製資料避免修改原始資料
train_df = train_df.copy()
test_df = test_df.copy()

# 步驟 1: 缺失值處理 - 刪除包含缺失值的列
train_df = train_df.dropna()
test_df = test_df.dropna()

# 步驟 2: 特徵移除
# 移除 education (與 education-num 重複) 和 fnlwgt (無關特徵)
columns_to_drop = ['education', 'fnlwgt']
train_df = train_df.drop(columns=columns_to_drop)
test_df = test_df.drop(columns=columns_to_drop)

# 步驟 3: 特徵工程 - 合併 native-country

# 3.1 合併 native-country
def merge_country(country):
    if country == 'United-States':
        return 'United-States'
    else:
        return 'Other'

train_df['native-country'] = train_df['native-country'].apply(merge_country)
test_df['native-country'] = test_df['native-country'].apply(merge_country)

# 3.2 創建資本收益特徵
train_df['capital'] = train_df['capital-gain'] - train_df['capital-loss']
test_df['capital'] = test_df['capital-gain'] - test_df['capital-loss']

# 3.4 創建年齡分組
def age_group(age):
    if age < 30:
        return 'young'
    elif age < 50:
        return 'middle'
    else:
        return 'senior'

train_df['age-group'] = train_df['age'].apply(age_group)
test_df['age-group'] = test_df['age'].apply(age_group)

# 步驟 4: 分離特徵和目標變數
# 目標變數是 hours-per-week
X_train = train_df.drop(columns=['hours-per-week','income'])
y_train = train_df['hours-per-week']
X_test = test_df.drop(columns=['hours-per-week','income'])
y_test = test_df['hours-per-week']


# 步驟 5: 定義數值和類別特徵

# 數值特徵
numeric_cols = ['age', 'education-num', 'capital-gain', 'capital-loss', 'capital']

# 類別特徵
categorical_cols = ['workclass', 'marital-status', 'occupation', 'relationship','race', 'sex', 'native-country','age-group']

# 數值特徵處理：填補缺失值（使用中位數）
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# 類別特徵處理：填補缺失值（使用眾數）+ One-Hot 編碼
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 組合前處理器
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# 執行前處理
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 取得特徵名稱
feature_names = (numeric_cols + 
                list(preprocessor.named_transformers_['cat']
                    .named_steps['onehot']
                    .get_feature_names_out(categorical_cols)))

print(f"\n前處理後的訓練集大小: {X_train_processed.shape}")
print(f"前處理後的測試集大小: {X_test_processed.shape}")

# 特徵標準化（為 SVR 準備）
print("\n" + "=" * 80)
print("特徵標準化 (For KNN/SVR)")
print("=" * 80)

# SVR 對特徵尺度非常敏感，需要標準化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_processed)
X_test_scaled = scaler.transform(X_test_processed)

print(f"\n特徵已標準化 (mean=0, std=1)")
print(f"標準化後的訓練集大小: {X_train_scaled.shape}")
print(f"標準化後的測試集大小: {X_test_scaled.shape}")

# ============================================================================
# 績效評估
# ============================================================================
def calculate_metrics(y_true, y_pred, dataset_name=""):
    """
    計算績效指標：MAPE, RMSE, R²
    """
    # MAPE (Mean Absolute Percentage Error)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    
    # RMSE (Root Mean Squared Error)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # R² (R-squared / Coefficient of Determination)
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n{dataset_name} 績效指標：")
    print(f"{'='*50}")
    print(f"MAPE (Mean Absolute Percentage Error): {mape:.4f}%")
    print(f"RMSE (Root Mean Squared Error):        {rmse:.4f}")
    print(f"R² (Coefficient of Determination):     {r2:.4f}")
    print(f"{'='*50}")
    
    return {'MAPE': mape, 'RMSE': rmse, 'R2': r2}

# ============================================================================
# 建立KNN 模型
# ============================================================================
print("\n" + "=" * 80)
print("訓練 KNN 模型")
print("=" * 80)

# 記錄訓練開始時間
knn_start_time = time.time()

# 建立 KNN 迴歸模型
knn_model = KNeighborsRegressor(
    n_neighbors=40,
    weights='uniform',
    algorithm='auto',
    n_jobs=-1
)

print("\n開始訓練...")
knn_model.fit(X_train_scaled, y_train)

# 記錄訓練結束時間
knn_training_time = time.time() - knn_start_time
print(f"\n訓練完成！耗時: {knn_training_time:.2f} 秒")

# ============================================================================
# KNN 模型預測
# ============================================================================
print("\n" + "=" * 80)
print("KNN 模型預測")
print("=" * 80)

# 記錄預測開始時間
knn_pred_start_time = time.time()

# 在訓練集上預測
y_train_pred_knn = knn_model.predict(X_train_scaled)

# 在測試集上預測
y_test_pred_knn = knn_model.predict(X_test_scaled)

# 記錄預測結束時間
knn_prediction_time = time.time() - knn_pred_start_time

print(f"\n預測完成！耗時: {knn_prediction_time:.2f} 秒")

print(f"\n【計算時間】")
print(f"  訓練時間: {knn_training_time:.2f} 秒")
print(f"  預測時間: {knn_prediction_time:.2f} 秒")
print(f"  總耗時: {knn_training_time + knn_prediction_time:.2f} 秒")
# ============================================================================
# KNN 績效評估
# ============================================================================
print("\n" + "=" * 80)
print("KNN 績效評估")
print("=" * 80)

# 訓練集績效
knn_train_metrics = calculate_metrics(y_train, y_train_pred_knn, "KNN 訓練集")

# 測試集績效
knn_test_metrics = calculate_metrics(y_test, y_test_pred_knn, "KNN 測試集")

# ============================================================================
# SVR 模型
# ============================================================================
print("\n" + "=" * 80)
print("SVR 模型訓練(SVR 訓練、預測時間較長，約6鐘，請稍後)")
print("=" * 80)

# 記錄訓練開始時間
svr_start_time = time.time()

# 建立 SVR 迴歸模型
svr_model = SVR(
    kernel='rbf',      # 使用 RBF (Radial Basis Function) 核函數
    C=10,              # 正則化參數
    epsilon=1.0,       # epsilon-tube 的寬度
    gamma='scale'      # RBF 核函數的係數
)

svr_model.fit(X_train_scaled, y_train)

# 記錄訓練結束時間
svr_training_time = time.time() - svr_start_time
print(f"\n訓練完成！耗時: {svr_training_time:.2f} 秒")

# ============================================================================
# SVR 模型預測
# ============================================================================
print("\n" + "=" * 80)
print("SVR 模型預測")
print("=" * 80)

# 記錄預測開始時間
svr_pred_start_time = time.time()

# 在訓練集上預測（使用標準化後的資料）
y_train_pred_svr = svr_model.predict(X_train_scaled)

# 在測試集上預測（使用標準化後的資料）
y_test_pred_svr = svr_model.predict(X_test_scaled)

# 記錄預測結束時間
svr_prediction_time = time.time() - svr_pred_start_time
print(f"\n預測完成！耗時: {svr_prediction_time:.2f} 秒")

print(f"\n【計算時間】")
print(f"  訓練時間: {svr_training_time:.2f} 秒")
print(f"  預測時間: {svr_prediction_time:.2f} 秒")
print(f"  總耗時: {svr_training_time + svr_prediction_time:.2f} 秒")
# ============================================================================
# SVR 績效評估
# ============================================================================
print("\n" + "=" * 80)
print("SVR 績效評估")
print("=" * 80)

# 訓練集績效
svr_train_metrics = calculate_metrics(y_train, y_train_pred_svr, "SVR 訓練集")

# 測試集績效
svr_test_metrics = calculate_metrics(y_test, y_test_pred_svr, "SVR 測試集")

# ============================================================================
# 建立 RandomForest 模型
# ============================================================================
print("\n" + "=" * 80)
print("RandomForest模型訓練")
print("=" * 80)
# 記錄訓練開始時間
start_time = time.time()

# 建立 RandomForest 迴歸模型
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rf_model.fit(X_train_processed, y_train)

# 記錄訓練結束時間
training_time = time.time() - start_time
print(f"\n訓練完成！耗時: {training_time:.2f} 秒")

# ============================================================================
# RandomForest模型預測
# ============================================================================
print("\n" + "=" * 80)
print("RandomForest模型預測")
print("=" * 80)

# 記錄預測開始時間
pred_start_time = time.time()

# 在訓練集上預測（用於評估訓練績效）
y_train_pred = rf_model.predict(X_train_processed)

# 在測試集上預測
y_test_pred = rf_model.predict(X_test_processed)

# 記錄預測結束時間
prediction_time = time.time() - pred_start_time
print(f"\n預測完成！耗時: {prediction_time:.2f} 秒")

print(f"\n【計算時間】")
print(f"  訓練時間: {training_time:.2f} 秒")
print(f"  預測時間: {prediction_time:.2f} 秒")
print(f"  總耗時: {training_time + prediction_time:.2f} 秒")
# ============================================================================
# RandomForest績效評估
# ============================================================================
print("\n" + "=" * 80)
print("RandomForest績效評估")
print("=" * 80)

# 訓練集績效
train_metrics = calculate_metrics(y_train, y_train_pred, "訓練集")

# 測試集績效
test_metrics = calculate_metrics(y_test, y_test_pred, "測試集")

# ============================================================================
# XGBoost 模型
# ============================================================================
print("\n" + "=" * 80)
print("XGBoost模型訓練")
print("=" * 80)
# 記錄訓練開始時間
xgb_start_time = time.time()

# 建立 XGBoost 迴歸模型
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,         
    colsample_bytree=0.8,  
    random_state=42,
    n_jobs=-1,
    verbosity=1
)
xgb_model.fit(X_train_processed, y_train)

# 記錄訓練結束時間
xgb_training_time = time.time() - xgb_start_time
print(f"\n訓練完成！耗時: {xgb_training_time:.2f} 秒")
# ============================================================================
# XGBoost 模型預測
# ============================================================================
print("\n" + "=" * 80)
print("XGBoost模型預測")
print("=" * 80)

# 記錄預測開始時間
xgb_pred_start_time = time.time()

# 在訓練集上預測
y_train_pred_xgb = xgb_model.predict(X_train_processed)

# 在測試集上預測
y_test_pred_xgb = xgb_model.predict(X_test_processed)

# 記錄預測結束時間
xgb_prediction_time = time.time() - xgb_pred_start_time
print(f"\n預測完成！耗時: {xgb_prediction_time:.2f} 秒")

print(f"\n【計算時間】")
print(f"  訓練時間: {xgb_training_time:.2f} 秒")
print(f"  預測時間: {xgb_prediction_time:.2f} 秒")
print(f"  總耗時: {xgb_training_time + xgb_prediction_time:.2f} 秒")
# ============================================================================
# XGBoost 績效評估
# ============================================================================
print("\n" + "=" * 80)
print("XGBoost 績效評估")
print("=" * 80)

# 訓練集績效
xgb_train_metrics = calculate_metrics(y_train, y_train_pred_xgb, "XGBoost 訓練集")

# 測試集績效
xgb_test_metrics = calculate_metrics(y_test, y_test_pred_xgb, "XGBoost 測試集")

# ============================================================================
# 四種演算法完整比較
# ============================================================================
print("\n" + "=" * 80)
print("迴歸模型演算法比較")
print("=" * 80)

# 建立比較表格
comparison_table = pd.DataFrame({
    '模型': ['KNN', 'SVR', 'RandomForest', 'XGBoost'],
    '訓練時間(秒)': [
        knn_training_time, 
        svr_training_time, 
        training_time, 
        xgb_training_time
    ],
    '預測時間(秒)': [
        knn_prediction_time, 
        svr_prediction_time, 
        prediction_time, 
        xgb_prediction_time
    ],
    '總耗時(秒)': [
        knn_training_time + knn_prediction_time,
        svr_training_time + svr_prediction_time,
        training_time + prediction_time,
        xgb_training_time + xgb_prediction_time
    ],
    'MAPE(%)': [
        knn_test_metrics['MAPE'], 
        svr_test_metrics['MAPE'], 
        test_metrics['MAPE'], 
        xgb_test_metrics['MAPE']
    ],
    'RMSE': [
        knn_test_metrics['RMSE'], 
        svr_test_metrics['RMSE'], 
        test_metrics['RMSE'], 
        xgb_test_metrics['RMSE']
    ],
    'R²': [
        knn_test_metrics['R2'], 
        svr_test_metrics['R2'], 
        test_metrics['R2'], 
        xgb_test_metrics['R2']
    ]
})

print(tabulate(comparison_table, headers='keys', tablefmt='grid', 
               floatfmt=('.0f', '.4f', '.4f', '.4f', '.4f', '.4f', '.4f'),
               showindex=False))
print("\n" + "=" * 80)
print("分析完成！")
print("=" * 80)

<>:36: SyntaxWarning: invalid escape sequence '\s'
<>:40: SyntaxWarning: invalid escape sequence '\s'
<>:36: SyntaxWarning: invalid escape sequence '\s'
<>:40: SyntaxWarning: invalid escape sequence '\s'
C:\Users\a8524\AppData\Local\Temp\ipykernel_24360\3830050850.py:36: SyntaxWarning: invalid escape sequence '\s'
  train_df = pd.read_csv(train_url, names=column_names, sep=',\s*',
C:\Users\a8524\AppData\Local\Temp\ipykernel_24360\3830050850.py:40: SyntaxWarning: invalid escape sequence '\s'
  test_df = pd.read_csv(test_url, names=column_names, sep=',\s*',


載入資料集...

原始訓練集大小: (32561, 15)
原始測試集大小: (16281, 15)

資料前置處理

前處理後的訓練集大小: (30162, 51)
前處理後的測試集大小: (15060, 51)

特徵標準化 (For KNN/SVR)

特徵已標準化 (mean=0, std=1)
標準化後的訓練集大小: (30162, 51)
標準化後的測試集大小: (15060, 51)

訓練 KNN 模型

開始訓練...

訓練完成！耗時: 0.00 秒

KNN 模型預測

預測完成！耗時: 3.19 秒

【計算時間】
  訓練時間: 0.00 秒
  預測時間: 3.19 秒
  總耗時: 3.19 秒

KNN 績效評估

KNN 訓練集 績效指標：
MAPE (Mean Absolute Percentage Error): 26.2657%
RMSE (Root Mean Squared Error):        10.3414
R² (Coefficient of Determination):     0.2548

KNN 測試集 績效指標：
MAPE (Mean Absolute Percentage Error): 27.6177%
RMSE (Root Mean Squared Error):        10.6730
R² (Coefficient of Determination):     0.2171

SVR 模型訓練(SVR 訓練、預測時間較長，約6鐘，請稍後)

訓練完成！耗時: 151.61 秒

SVR 模型預測
